# ☕ Barista Bot RAG — V1.0

Bem-vindo ao **Barista Bot**, um assistente de perguntas e respostas que responde **exclusivamente** com base no conteúdo de um livro de café — mesmo quando esse livro existe apenas como um **PDF escaneado (imagem)**.

### O detalhe crítico dos dados

Nossa única fonte de verdade é um PDF **escaneado**, ou seja, sem camada de texto pesquisável. Isso significa que bibliotecas tradicionais como `PyPDF2` ou `pdfplumber` **não funcionam** — elas leem texto embutido no PDF, e aqui não existe texto algum, apenas imagens de páginas.

A solução é tratar cada página como uma **imagem** e aplicar **OCR (Optical Character Recognition)** para extrair o texto. Este notebook implementa esse pipeline do zero, com rastreabilidade de página (metadado `page`), e monta em cima dele uma arquitetura RAG completa.

### O que você vai encontrar aqui

1. **Setup** — dependências de sistema (Tesseract OCR + idioma português) e bibliotecas Python.
2. **Ollama** — subimos um servidor LLM local (`llama3`) em background, direto no ambiente do Colab.
3. **Ingestão via OCR** — conversão do PDF em imagens e extração de texto, com metadado de página.
4. **Vetorização** — chunking, embeddings e armazenamento em um índice FAISS.
5. **RAG Chain** — um prompt que **proíbe** a IA de usar conhecimento externo ao livro.
6. **Interação** — perguntas e respostas citando a página de origem.

> 🔮 Esta é a **V1.0**: um pipeline RAG linear e determinístico. A **V2.0** irá evoluir este mesmo pipeline para uma arquitetura **agêntica** (veja o README do repositório para o roadmap).

## 0. Ambiente Colab — Clonar o repositório e localizar o PDF

Esta célula prepara o ambiente do zero, do jeito que o Google Colab entrega uma máquina limpa a cada sessão:

1. **Clona este repositório do GitHub** para dentro de `/content`, trazendo o notebook, o README e o `.gitignore` — mas **não** o PDF do livro (ele fica de fora do Git de propósito, por ser um arquivo de dados).
2. **Monta o Google Drive**, onde você deve manter o PDF escaneado do livro salvo (evita ter que fazer upload manual a cada sessão).
3. **Copia o PDF do Drive** para dentro da pasta `data/` do repositório clonado, que é onde a função `carregar_documentos()` (mais abaixo) vai procurá-lo.

> ✏️ **Antes de rodar**, edite a variável `PASTA_PDF_NO_DRIVE` na célula abaixo, trocando o placeholder pelo caminho real da pasta do seu Drive onde está o PDF. Essa edição é só local — não é necessário (nem recomendado) commitar seu caminho pessoal do Drive de volta no repositório público.

In [ ]:
import os
import shutil

REPO_URL = "https://github.com/wdasilvamf/barista-bot-rag.git"
REPO_DIR = "/content/barista-bot-rag"

# 1. Clona o repositório (só se ainda não existir nesta sessão)
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repositório já clonado nesta sessão.")

%cd {REPO_DIR}

# 2. Monta o Google Drive
from google.colab import drive
drive.mount("/content/drive")

# 3. Copia o PDF do Drive para a pasta data/ do repositório
#    <<< EDITE AQUI: troque pelo caminho real da pasta no SEU Google Drive >>>
PASTA_PDF_NO_DRIVE = "/content/drive/MyDrive/SEU_CAMINHO_AQUI"
PASTA_DATA_DESTINO = os.path.join(REPO_DIR, "data")

pdfs_copiados = []
for arquivo in os.listdir(PASTA_PDF_NO_DRIVE):
    if arquivo.lower().endswith(".pdf"):
        shutil.copy(os.path.join(PASTA_PDF_NO_DRIVE, arquivo), PASTA_DATA_DESTINO)
        pdfs_copiados.append(arquivo)

if pdfs_copiados:
    print(f"✅ PDF(s) copiado(s) para {PASTA_DATA_DESTINO}: {pdfs_copiados}")
else:
    print(f"⚠️ Nenhum PDF encontrado em '{PASTA_PDF_NO_DRIVE}'. Verifique o caminho.")

## 1. Setup — Dependências de sistema e Python

Como o pipeline depende de **OCR**, precisamos instalar via `apt-get` o motor `tesseract-ocr` e o pacote de idioma **português** (`tesseract-ocr-por`) — sem ele, o Tesseract não reconhece acentuação e palavras em PT-BR corretamente.

Também instalamos o `poppler-utils`, dependência de sistema exigida pelo `pdf2image` para rasterizar páginas de PDF em imagens.

In [ ]:
# --- Dependências de SISTEMA (via apt-get) ---
# tesseract-ocr        -> motor de OCR
# tesseract-ocr-por    -> pacote de idioma Português para o Tesseract
# poppler-utils        -> necessário para o pdf2image converter PDF -> imagem
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por poppler-utils

# --- Dependências PYTHON ---
!pip install -q pytesseract pdf2image langchain langchain-community \
    faiss-cpu sentence-transformers huggingface_hub pypdf

## 2. Ollama — Subindo o LLM local em background

Usamos o [Ollama](https://ollama.com/) para rodar o modelo **`llama3`** localmente dentro do ambiente do Colab, evitando dependência de APIs pagas externas.

O processo abaixo:
1. Instala o Ollama via script oficial.
2. Sobe o servidor Ollama (`ollama serve`) em **background**, usando `subprocess.Popen`, para que a célula não fique bloqueada esperando o servidor encerrar.
3. Aguarda o servidor ficar disponível.
4. Faz o `pull` do modelo `llama3`.

In [ ]:
import subprocess
import time
import requests

# 1. Instala o Ollama (script oficial)
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Sobe o servidor Ollama em background.
#    Usamos Popen (não subprocess.run) porque 'ollama serve' é um processo
#    de longa duração: run() ficaria bloqueado esperando ele terminar.
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# 3. Aguarda o servidor responder antes de prosseguir
OLLAMA_URL = "http://localhost:11434"
for tentativa in range(30):
    try:
        if requests.get(OLLAMA_URL).status_code == 200:
            print("✅ Servidor Ollama no ar.")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama não respondeu a tempo. Verifique a instalação.")

# 4. Baixa o modelo llama3 (pode levar alguns minutos na primeira execução)
!ollama pull llama3

## 3. Ingestão de Dados — OCR do PDF escaneado

Esta é a etapa **core** do projeto. Como o PDF é uma sequência de páginas escaneadas (imagens), o fluxo é:

1. **`pdf2image.convert_from_path`** converte cada página do PDF em uma imagem (`PIL.Image`).
2. **`pytesseract.image_to_string`** aplica OCR em cada imagem, usando `lang="por"` para reconhecer português corretamente.
3. Cada trecho de texto extraído é embrulhado em um `Document` do LangChain, com metadado `{"page": N}` — isso é **vital**: é o que permite ao usuário final saber exatamente de qual página do livro veio cada resposta.
4. O texto completo é salvo em cache (`data/ocr_cache.json`) para não precisarmos reprocessar o PDF inteiro (processo lento) toda vez que o notebook é executado.

In [ ]:
import os
import json
from pdf2image import convert_from_path
import pytesseract
from langchain.docstore.document import Document

DATA_DIR = "/content/data" if os.path.isdir("/content") else "data"
CACHE_PATH = os.path.join(DATA_DIR, "ocr_cache.json")


def encontrar_pdf(data_dir: str) -> str:
    """Localiza o primeiro PDF dentro da pasta de dados."""
    pdfs = [f for f in os.listdir(data_dir) if f.lower().endswith(".pdf")]
    if not pdfs:
        raise FileNotFoundError(
            f"Nenhum PDF encontrado em '{data_dir}'. Faça upload do livro escaneado nessa pasta."
        )
    return os.path.join(data_dir, pdfs[0])


def extrair_texto_via_ocr(caminho_pdf: str, idioma: str = "por", dpi: int = 300) -> list[Document]:
    """Converte cada página do PDF escaneado em imagem e aplica OCR.

    Retorna uma lista de Documents do LangChain, um por página, cada um com
    metadado {'page': numero_da_pagina, 'source': nome_do_arquivo}. Esse
    metadado de página é o que permite, mais tarde, informar ao usuário de
    onde no livro cada resposta foi extraída.
    """
    print(f"📄 Convertendo páginas de '{caminho_pdf}' em imagens (dpi={dpi})...")
    paginas_imagem = convert_from_path(caminho_pdf, dpi=dpi)

    documentos = []
    for numero_pagina, imagem in enumerate(paginas_imagem, start=1):
        print(f"🔎 OCR na página {numero_pagina}/{len(paginas_imagem)}...", end="\r")
        texto = pytesseract.image_to_string(imagem, lang=idioma)
        texto = texto.strip()

        if not texto:
            # Página sem texto reconhecível (ex: capa em branco) -> ignora
            continue

        documentos.append(
            Document(
                page_content=texto,
                metadata={
                    "page": numero_pagina,
                    "source": os.path.basename(caminho_pdf),
                },
            )
        )
    print(f"\n✅ OCR concluído: {len(documentos)} páginas com texto extraído.")
    return documentos


def carregar_documentos(data_dir: str = DATA_DIR, forcar_reprocessamento: bool = False) -> list[Document]:
    """Carrega os documentos do cache se existir; caso contrário, roda o OCR."""
    if os.path.exists(CACHE_PATH) and not forcar_reprocessamento:
        print("⚡ Cache de OCR encontrado. Carregando texto já extraído...")
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            bruto = json.load(f)
        return [Document(page_content=d["page_content"], metadata=d["metadata"]) for d in bruto]

    caminho_pdf = encontrar_pdf(data_dir)
    documentos = extrair_texto_via_ocr(caminho_pdf)

    # Salva cache para acelerar execuções futuras
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(
            [{"page_content": d.page_content, "metadata": d.metadata} for d in documentos],
            f,
            ensure_ascii=False,
        )
    return documentos


documentos = carregar_documentos()
print(f"\nExemplo (página {documentos[0].metadata['page']}):\n")
print(documentos[0].page_content[:500])

## 4. Vetorização — Chunking, Embeddings e FAISS

Com o texto extraído (e a página de origem preservada em cada `Document`), seguimos o pipeline RAG padrão:

1. **`RecursiveCharacterTextSplitter`** quebra o texto de cada página em pedaços menores (*chunks*), preservando o metadado `page` de origem em cada chunk gerado.
2. **Embeddings** — usamos o modelo `BAAI/bge-small-en-v1.5` da HuggingFace, leve e eficiente para rodar em CPU.
3. **FAISS** — armazenamos os vetores em um índice FAISS local, permitindo busca por similaridade semântica.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Chunking: quebra páginas longas em pedaços menores e mais coesos
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(documentos)
print(f"📚 {len(documentos)} páginas divididas em {len(chunks)} chunks.")

# 2. Embeddings: modelo leve, roda bem em CPU
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

# 3. Índice vetorial FAISS
vector_store = FAISS.from_documents(chunks, embeddings)
print("✅ Índice FAISS criado com sucesso.")

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

## 5. RAG Chain — Prompt restritivo + Ollama (llama3)

Aqui montamos a *chain* que liga o `retriever` (FAISS) ao LLM (`llama3` via Ollama).

O ponto central desta etapa é o **prompt template**: ele instrui explicitamente o modelo a **responder apenas com base no contexto recuperado do livro**, proibindo o uso de conhecimento externo. Isso reduz drasticamente o risco de alucinação e garante que o Barista Bot seja fiel à fonte.

In [ ]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

llm = Ollama(model="llama3", base_url=OLLAMA_URL, temperature=0)

# Prompt restritivo: a IA só pode responder com base no CONTEXTO fornecido.
# Isso impede o modelo de "completar" respostas com conhecimento externo
# ao livro, que é a nossa única fonte de verdade.
PROMPT_TEMPLATE = """Você é o Barista Bot, um especialista em café que responde
EXCLUSIVAMENTE com base no CONTEXTO abaixo, extraído de um livro sobre café.

Regras estritas:
- NÃO utilize nenhum conhecimento externo ao CONTEXTO, mesmo que você o conheça.
- Se a resposta não estiver no CONTEXTO, diga claramente:
  "Não encontrei essa informação no livro."
- Responda em português, de forma clara e objetiva.

CONTEXTO:
{context}

PERGUNTA:
{question}

RESPOSTA (baseada apenas no CONTEXTO):"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)
print("✅ RAG chain pronta.")

## 6. Interação — Pergunte ao Barista Bot

A função abaixo executa a chain e imprime, além da resposta, as **páginas do livro** que embasaram a resposta — cumprindo o requisito de rastreabilidade da fonte.

In [ ]:
def perguntar(pergunta: str):
    resultado = rag_chain.invoke({"query": pergunta})
    resposta = resultado["result"]
    fontes = sorted({doc.metadata.get("page") for doc in resultado["source_documents"]})

    print("🤖 Resposta:\n")
    print(resposta)
    print("\n📖 Fonte(s): página(s)", ", ".join(str(p) for p in fontes if p is not None))
    return resultado


# Exemplo de uso:
_ = perguntar("Qual a temperatura ideal da água para o preparo do café?")

---
### ✅ Fim da V1.0

Você acabou de construir um pipeline RAG completo, do PDF escaneado até a resposta rastreável por página, usando **apenas** ferramentas open-source rodando localmente (Tesseract, FAISS, Ollama).

Próximo passo do projeto: evoluir este pipeline linear para uma **arquitetura agêntica (V2.0)** — veja o `README.md` do repositório para o roadmap detalhado.